# Etape 1

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MiniProjet_Spark").master("local[2]").getOrCreate()
print("Master =", spark.sparkContext.master)

Master = local[2]


### 1. Quel est le rôle de Docker dans ce mini-projet ?
Docker nous permet de lancer Spark et Jupyter dans un conteneur, sans avoir à installer toutes les dépendances sur nos ordinateurs.

### 2. Pourquoi utilisons-nous une image Docker contenant PySpark ?
L'image contient déjà Python, PySpark et Jupyter. On peut donc directement travailler avec Spark sans devoir tout configurer nous-mêmes.

### 3. Quelle est la différence entre un conteneur Docker et une machine virtuelle ?
Un conteneur Docker partage le noyau du système sur lequel il tourne, alors qu'une machine virtuelle possède son propre système d'exploitation et son propre noyau. Docker est donc généralement plus léger et plus rapide à démarrer.

### 4. À quoi sert le fichier docker-compose.yml ?
Il sert à configurer et lancer notre conteneur. Dans notre cas, il précise l'image PySpark à utiliser, le port pour accéder à Jupyter et le dossier partagé entre notre ordinateur et le conteneur.

### 5. Que signifie le mode d'exécution local[2] de Spark ?
Cela signifie que Spark tourne sur une seule machine avec 2 threads, ce qui lui permet d'exécuter jusqu'à 2 tâches en parallèle.

### 6. Quel est le rôle de SparkSession ?
C'est le point d'entrée de Spark. On l'utilise pour travailler avec les DataFrames, charger des données et lancer des requêtes SQL.


# Etape 2

In [16]:
import os
from pyspark.sql.functions import col
print(os.listdir("/home/jovyan/work"))

['output', 'MiniProject_Spark.ipynb', 'purchases.txt', 'notebooks']


In [6]:
with open("/home/jovyan/work/purchases.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().rstrip())

2012-01-01	09:00	San Jose	Men's Clothing	214.05	Amex
2012-01-01	09:00	Fort Worth	Women's Clothing	153.57	Visa
2012-01-01	09:00	San Diego	Music	66.08	Cash
2012-01-01	09:00	Pittsburgh	Pet Supplies	493.51	Discover
2012-01-01	09:00	Omaha	Children's Clothing	235.63	MasterCard


In [29]:
# 1. Charger les données dans un DataFrame
df = spark.read.csv(
    "/home/jovyan/work/purchases.txt",
    sep="\t",
    header=False
)

In [30]:
# 2. Définir les noms des colonnes
print(df.columns)
df = df.toDF("Date", "Heure", "Magasin", "Categorie", "Cout", "Paiement")
print(df.columns)

['_c0', '_c1', '_c2', '_c3', '_c4', '_c5']
['Date', 'Heure', 'Magasin', 'Categorie', 'Cout', 'Paiement']


In [31]:
# 3. Afficher les 10 premières lignes
df.show(10)

+----------+-----+--------------+--------------------+------+----------+
|      Date|Heure|       Magasin|           Categorie|  Cout|  Paiement|
+----------+-----+--------------+--------------------+------+----------+
|2012-01-01|09:00|      San Jose|      Men's Clothing|214.05|      Amex|
|2012-01-01|09:00|    Fort Worth|    Women's Clothing|153.57|      Visa|
|2012-01-01|09:00|     San Diego|               Music| 66.08|      Cash|
|2012-01-01|09:00|    Pittsburgh|        Pet Supplies|493.51|  Discover|
|2012-01-01|09:00|         Omaha| Children's Clothing|235.63|MasterCard|
|2012-01-01|09:00|      Stockton|      Men's Clothing|247.18|MasterCard|
|2012-01-01|09:00|        Austin|             Cameras| 379.6|      Visa|
|2012-01-01|09:00|      New York|Consumer Electronics| 296.8|      Cash|
|2012-01-01|09:00|Corpus Christi|                Toys| 25.38|  Discover|
|2012-01-01|09:00|    Fort Worth|                Toys|213.88|      Visa|
+----------+-----+--------------+------------------

In [32]:
# 4. Afficher le schéma du DataFrame
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Heure: string (nullable = true)
 |-- Magasin: string (nullable = true)
 |-- Categorie: string (nullable = true)
 |-- Cout: string (nullable = true)
 |-- Paiement: string (nullable = true)



In [33]:
# 5. Vérifier et convertir Cout en numerique
print(f"Cout type : {df.schema["Cout"].dataType}")

df = df.withColumn("Cout", col("Cout").cast("double"))

df.printSchema()


Cout type : StringType()
root
 |-- Date: string (nullable = true)
 |-- Heure: string (nullable = true)
 |-- Magasin: string (nullable = true)
 |-- Categorie: string (nullable = true)
 |-- Cout: double (nullable = true)
 |-- Paiement: string (nullable = true)



In [34]:
# 6. Nombre total de transactions
nb_transactions = df.count()

print("Nombre total de transactions :", nb_transactions)

Nombre total de transactions : 4138476


# Etape 3

# Etape 4

In [26]:
from pyspark.sql.functions import sum, avg

# 1. Afficher uniquement les colonnes Magasin, Categorie et Cout
df.select("Magasin", "Categorie", "Cout").show(10)

+--------------+--------------------+------+
|       Magasin|           Categorie|  Cout|
+--------------+--------------------+------+
|      San Jose|      Men's Clothing|214.05|
|    Fort Worth|    Women's Clothing|153.57|
|     San Diego|               Music| 66.08|
|    Pittsburgh|        Pet Supplies|493.51|
|         Omaha| Children's Clothing|235.63|
|      Stockton|      Men's Clothing|247.18|
|        Austin|             Cameras| 379.6|
|      New York|Consumer Electronics| 296.8|
|Corpus Christi|                Toys| 25.38|
|    Fort Worth|                Toys|213.88|
+--------------+--------------------+------+
only showing top 10 rows


In [27]:
# 2. Filtrer les ventes dont le montant est supérieur à 100
df.filter(col("Cout") > 100).show(10)

+----------+-----+----------+--------------------+------+----------+
|      Date|Heure|   Magasin|           Categorie|  Cout|  Paiement|
+----------+-----+----------+--------------------+------+----------+
|2012-01-01|09:00|  San Jose|      Men's Clothing|214.05|      Amex|
|2012-01-01|09:00|Fort Worth|    Women's Clothing|153.57|      Visa|
|2012-01-01|09:00|Pittsburgh|        Pet Supplies|493.51|  Discover|
|2012-01-01|09:00|     Omaha| Children's Clothing|235.63|MasterCard|
|2012-01-01|09:00|  Stockton|      Men's Clothing|247.18|MasterCard|
|2012-01-01|09:00|    Austin|             Cameras| 379.6|      Visa|
|2012-01-01|09:00|  New York|Consumer Electronics| 296.8|      Cash|
|2012-01-01|09:00|Fort Worth|                Toys|213.88|      Visa|
|2012-01-01|09:00|    Austin|             Cameras|469.63|MasterCard|
|2012-01-01|09:00|Greensboro|                DVDs|290.82|MasterCard|
+----------+-----+----------+--------------------+------+----------+
only showing top 10 rows


In [28]:
# 3. Trier ces ventes par montant décroissant
df.filter(col("Cout") > 100).orderBy(col("Cout").desc()).show(10)

+----------+-----+-----------+--------------------+------+----------+
|      Date|Heure|    Magasin|           Categorie|  Cout|  Paiement|
+----------+-----+-----------+--------------------+------+----------+
|2012-08-27|17:02|Los Angeles|Consumer Electronics|499.99|  Discover|
|2012-10-07|15:19|    Phoenix| Children's Clothing|499.99|      Amex|
|2012-09-08|12:33|    Detroit|              Crafts|499.99|MasterCard|
|2012-08-13|16:47|      Plano|         Video Games|499.99|      Amex|
|2012-09-12|12:28|      Omaha|           Computers|499.99|  Discover|
|2012-07-12|17:54|     Boston|              Crafts|499.99|      Cash|
|2012-09-17|09:08|  St. Louis|               Books|499.99|      Cash|
|2012-07-08|09:49|     Dallas|                 CDs|499.99|MasterCard|
|2012-09-18|09:45| Pittsburgh|              Garden|499.99|      Amex|
|2012-08-10|14:48|   Honolulu|   Health and Beauty|499.99|  Discover|
+----------+-----+-----------+--------------------+------+----------+
only showing top 10 

In [42]:
# 4. Calculer le chiffre d'affaires total par magasin
df_ca_magasin = df.groupBy("Magasin").agg(sum("Cout").alias("CA_total"))

df_ca_magasin.show()

+---------------+--------------------+
|        Magasin|            CA_total|
+---------------+--------------------+
|North Las Vegas|1.0029652509999933E7|
|        Phoenix|1.0079076700000003E7|
|          Omaha| 1.002664233999998E7|
|      Anchorage|   9933500.399999995|
|        Anaheim|1.0076416359999968E7|
|     Greensboro|1.0033781390000068E7|
|         Dallas|1.0066548450000022E7|
|        Oakland|   9947292.520000033|
|         Laredo|1.0144604979999993E7|
|     Scottsdale|1.0037929849999957E7|
|    San Antonio|1.0014441700000005E7|
|    Bakersfield|1.0031208919999965E7|
|        Raleigh|1.0061442539999988E7|
|    Chula Vista|   9974951.340000018|
|   Philadelphia|1.0190080259999951E7|
|     Louisville|1.0008566470000014E7|
|    Los Angeles|1.0084576799999982E7|
|       Chandler|   9919559.859999968|
|     Sacramento|1.0123468179999966E7|
|   Indianapolis|1.0090272770000013E7|
+---------------+--------------------+
only showing top 20 rows


In [39]:
# 5. Identifier le magasin ayant le chiffre d'affaires le plus élevé
df_ca_magasin.orderBy(col("CA_total").desc()).show(1)

+------------+--------------------+
|     Magasin|            CA_total|
+------------+--------------------+
|Philadelphia|1.0190080259999951E7|
+------------+--------------------+
only showing top 1 row


In [41]:
# 6. Calculer le montant moyen des transactions par catégorie de produits
df.groupBy("Categorie").agg(avg("Cout").alias("Montant_moyen")).show()

+--------------------+------------------+
|           Categorie|     Montant_moyen|
+--------------------+------------------+
| Children's Clothing|250.03285014470467|
|      Sporting Goods|250.50487052693825|
|                 CDs|249.56965140693512|
|           Computers|250.22114966013262|
|Consumer Electronics|250.05276844198997|
|   Health and Beauty| 250.2823198805272|
|        Pet Supplies|249.52775143746985|
|                DVDs| 250.3505047899475|
|                Baby| 249.6463567715953|
|              Crafts|249.91688538361313|
|    Women's Clothing| 249.6607214518553|
|         Video Games| 249.7998392091628|
|               Books| 250.0174418483193|
|               Music|249.81746556593654|
|      Men's Clothing|250.05979707503488|
|             Cameras|249.86502110588287|
|              Garden|250.09380983426973|
|                Toys|249.88031652780342|
+--------------------+------------------+



# Etape 5

In [43]:
# 1. Créer une vue temporaire nommée ventes
df.createOrReplaceTempView("ventes")

In [44]:
# 2. Afficher les ventes dont le montant dépasse 100
spark.sql("""
    SELECT *
    FROM ventes
    WHERE Cout > 100
""").show(10)

+----------+-----+----------+--------------------+------+----------+
|      Date|Heure|   Magasin|           Categorie|  Cout|  Paiement|
+----------+-----+----------+--------------------+------+----------+
|2012-01-01|09:00|  San Jose|      Men's Clothing|214.05|      Amex|
|2012-01-01|09:00|Fort Worth|    Women's Clothing|153.57|      Visa|
|2012-01-01|09:00|Pittsburgh|        Pet Supplies|493.51|  Discover|
|2012-01-01|09:00|     Omaha| Children's Clothing|235.63|MasterCard|
|2012-01-01|09:00|  Stockton|      Men's Clothing|247.18|MasterCard|
|2012-01-01|09:00|    Austin|             Cameras| 379.6|      Visa|
|2012-01-01|09:00|  New York|Consumer Electronics| 296.8|      Cash|
|2012-01-01|09:00|Fort Worth|                Toys|213.88|      Visa|
|2012-01-01|09:00|    Austin|             Cameras|469.63|MasterCard|
|2012-01-01|09:00|Greensboro|                DVDs|290.82|MasterCard|
+----------+-----+----------+--------------------+------+----------+
only showing top 10 rows


In [49]:
# 3. Calculer le chiffre d'affaires total par magasin
df_ca_sql = spark.sql("""
    SELECT
        Magasin,
        SUM(Cout) AS CA_total
    FROM ventes
    GROUP BY Magasin
""")

df_ca_sql.show()

+---------------+--------------------+
|        Magasin|            CA_total|
+---------------+--------------------+
|North Las Vegas|1.0029652509999933E7|
|        Phoenix|1.0079076700000003E7|
|          Omaha| 1.002664233999998E7|
|      Anchorage|   9933500.399999995|
|        Anaheim|1.0076416359999968E7|
|     Greensboro|1.0033781390000068E7|
|         Dallas|1.0066548450000022E7|
|        Oakland|   9947292.520000033|
|         Laredo|1.0144604979999993E7|
|     Scottsdale|1.0037929849999957E7|
|    San Antonio|1.0014441700000005E7|
|    Bakersfield|1.0031208919999965E7|
|        Raleigh|1.0061442539999988E7|
|    Chula Vista|   9974951.340000018|
|   Philadelphia|1.0190080259999951E7|
|     Louisville|1.0008566470000014E7|
|    Los Angeles|1.0084576799999982E7|
|       Chandler|   9919559.859999968|
|     Sacramento|1.0123468179999966E7|
|   Indianapolis|1.0090272770000013E7|
+---------------+--------------------+
only showing top 20 rows


In [48]:
# 4. Afficher les trois magasins ayant le chiffre d'affaires le plus élevé
spark.sql("""
    SELECT
        Magasin,
        SUM(Cout) AS CA_total
    FROM ventes
    GROUP BY Magasin
    ORDER BY CA_total DESC
    LIMIT 3
""").show()

+------------+--------------------+
|     Magasin|            CA_total|
+------------+--------------------+
|Philadelphia|1.0190080259999951E7|
|      Durham|1.0153890210000012E7|
|      Laredo|1.0144604979999993E7|
+------------+--------------------+



### 5. Comparaison entre Spark SQL et DataFrame API

Les deux méthodes donnent les mêmes résultats. Avec la DataFrame API, on utilise des fonctions Python comme `filter()` ou `groupBy()`, alors qu'avec Spark SQL, on écrit directement des requêtes SQL.

Par exemple, pour afficher les ventes supérieures à 100 :

**DataFrame API :**
```python
df.filter(col("Cout") > 100).show()
```

**Spark SQL :**
```sql
SELECT * FROM ventes WHERE Cout > 100
```

C'est surtout la syntaxe qui change donc, puisque les traitements sont exécutés par le même moteur Spark.


# Etape 6

### Sauvegarde des résultats

In [52]:
# 1. Récupérer le DataFrame contenant le chiffre d'affaires par magasin
df_resultat = df_ca_magasin

df_resultat.show(10)

+---------------+--------------------+
|        Magasin|            CA_total|
+---------------+--------------------+
|North Las Vegas|1.0029652509999933E7|
|        Phoenix|1.0079076700000003E7|
|          Omaha| 1.002664233999998E7|
|      Anchorage|   9933500.399999995|
|        Anaheim|1.0076416359999968E7|
|     Greensboro|1.0033781390000068E7|
|         Dallas|1.0066548450000022E7|
|        Oakland|   9947292.520000033|
|         Laredo|1.0144604979999993E7|
|     Scottsdale|1.0037929849999957E7|
+---------------+--------------------+
only showing top 10 rows


In [55]:
# 2. Enregistrer les résultats au format Parquet avec overwrite
output_path = "/home/jovyan/work/output/vente_par_magasin"

df_resultat.write.mode("overwrite").parquet(output_path)

In [56]:
# 3. Vérifier que les fichiers Parquet ont été créés
print(os.listdir(output_path))

['part-00000-27084eb1-8021-4a89-be86-3c0783124b5d-c000.snappy.parquet', '._SUCCESS.crc', '_SUCCESS', '.part-00000-27084eb1-8021-4a89-be86-3c0783124b5d-c000.snappy.parquet.crc']


### Vérification des données enregistrées

In [58]:
# 1. Relire les données Parquet dans une nouvelle DataFrame
df_verification = spark.read.parquet("/home/jovyan/work/output/vente_par_magasin")


In [59]:
# 2. Afficher les premières lignes
df_verification.show(5)

+---------------+--------------------+
|        Magasin|            CA_total|
+---------------+--------------------+
|North Las Vegas|1.0029652509999933E7|
|        Phoenix|1.0079076700000003E7|
|          Omaha| 1.002664233999998E7|
|      Anchorage|   9933500.399999995|
|        Anaheim|1.0076416359999968E7|
+---------------+--------------------+
only showing top 5 rows
